In [1]:
import random

class BatchGenerator:
    def __init__(self, data, batch_size, shuffle=True):
        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.index = 0
        self._prepare_data()

    def _prepare_data(self):
        if self.shuffle:
            random.shuffle(self.data)
        self.index = 0

    def next_batch(self):
        # Check if we need to reset/shuffle
        if self.index >= len(self.data):
            self._prepare_data()

        # Get the batch slice
        end_index = min(self.index + self.batch_size, len(self.data))
        batch = self.data[self.index:end_index]
        self.index = end_index
        
        return batch

# Test case
generator = BatchGenerator([1, 2, 3, 4, 5], batch_size=2, shuffle=False)
print(generator.next_batch()) # Expected: [1, 2]
print(generator.next_batch()) # Expected: [3, 4]
print(generator.next_batch()) # Expected: [5]
print(generator.next_batch()) # Expected: [1, 2] (loops back)

[1, 2]
[3, 4]
[5]
[1, 2]


In [2]:
import random
import math

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def kmeans(data, k, max_iters=100):
    # 1. Randomly initialize centroids from the data points
    centroids = random.sample(data, k)
    
    for _ in range(max_iters):
        # Create empty clusters
        clusters = [[] for _ in range(k)]
        
        # 2. Assign each point to the closest centroid
        for point in data:
            distances = [euclidean_distance(point, c) for c in centroids]
            closest_index = distances.index(min(distances))
            clusters[closest_index].append(point)
            
        # 3. Store old centroids to check for convergence
        prev_centroids = centroids[:]
        
        # 4. Update centroids to be the mean of their assigned points
        for i, cluster in enumerate(clusters):
            if not cluster: # Handle edge case: empty cluster
                continue
            mean_x = sum(p[0] for p in cluster) / len(cluster)
            mean_y = sum(p[1] for p in cluster) / len(cluster)
            centroids[i] = [mean_x, mean_y]
            
        # 5. Check for convergence (if centroids didn't move)
        if centroids == prev_centroids:
            break
            
    return centroids, clusters

# Test case
data_points = [[1, 2], [1, 4], [1, 0], [10, 2], [10, 4], [10, 0]]
final_centroids, final_clusters = kmeans(data_points, k=2)
print("Centroids:", final_centroids)
print("final_clusters:", final_clusters)

Centroids: [[5.5, 3.0], [5.5, 0.0]]
final_clusters: [[[1, 2], [1, 4], [10, 2], [10, 4]], [[1, 0], [10, 0]]]


In [3]:
def softmax(logits):
    max_logits = max(logits)

    exp = [math.exp(x - max_logits) for x in logits]
    sum_exp = sum(exp)

    return [x / sum_exp for x in exp]

print(softmax([1.0, 2.0, 3.0])) 
# -> [0.090, 0.244, 0.665]

[0.09003057317038046, 0.24472847105479764, 0.6652409557748218]


In [4]:
import math
from collections import Counter

def knn_predict(X_train, y_train, test_point, k=3):
    distances = []
    
    # 1. Calculate Euclidean distance from test_point to all training points
    for i, train_point in enumerate(X_train):
        # Handles n-dimensional points using zip
        dist = math.sqrt(sum((a - b) ** 2 for a, b in zip(train_point, test_point)))
        distances.append((dist, y_train[i]))
        
    # 2. Sort by distance (ascending)
    distances.sort(key=lambda x: x[0])
    
    # 3. Get the labels of the top K nearest neighbors
    top_k_labels = [label for _, label in distances[:k]]
    
    # 4. Return the most common label (majority vote)
    return Counter(top_k_labels).most_common(1)[0][0]

# Test:
X = [[1, 2], [1.5, 1.8], [5, 8], [8, 8]]
y = [0, 0, 1, 1]
print(knn_predict(X, y, [2, 2], k=3)) 
# -> 0

0


In [5]:
import math
def bce_loss(ytrue, ypred):
    n = len(ytrue)
    epsilon = 1e-15

    # -1/n x ylog(yhat) + (1-y)log(1-yhat)
    loss = 0

    for yt, yp in zip(ytrue, ypred):
        yp = max(epsilon, min(1 - epsilon, yp))
        loss += -((yt * math.log(yp)) + (1 - yt) * math.log(1- yp))
    
    return loss/n

# def bce_loss(y_true, y_pred):
#     epsilon = 1e-15 # Small value to prevent log(0)
#     loss = 0
#     n = len(y_true)
    
#     for yt, yp in zip(y_true, y_pred):
#         # Clip the prediction to be strictly between epsilon and 1-epsilon
#         yp = max(epsilon, min(1 - epsilon, yp))
        
#         # Calculate loss for this instance
#         loss += -(yt * math.log(yp) + (1 - yt) * math.log(1 - yp))
        
#     return loss / n

# Test: print(bce_loss([1, 0, 1], [0.9, 0.1, 0.8])) -> ~0.164
print(bce_loss([1, 0, 1], [0.9, 0.1, 0.8])) 
# -> ~0.164

0.14462152754328741


In [6]:
def calculate_metrics(y_true, y_pred):
    tp = fp = fn = 0
    
    # 1. Calculate TP, FP, FN
    for yt, yp in zip(y_true, y_pred):
        if yt == 1 and yp == 1:
            tp += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
            
    # 2. Calculate Precision (handling division by zero)
    # Precision = TP / (TP + FP)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    
    # 3. Calculate Recall (handling division by zero)
    # Recall = TP / (TP + FN)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    # 4. Calculate F1 Score
    # F1 = 2 * (Precision * Recall) / (Precision + Recall)
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0.0
        
    return precision, recall, f1

# Test case
y_true = [1, 0, 1, 1, 0, 1]
y_pred = [1, 0, 1, 0, 0, 1]
print(calculate_metrics(y_true, y_pred)) 
# Expected TP=3, FP=0, FN=1 -> Precision=1.0, Recall=0.75, F1=0.857...

(1.0, 0.75, 0.8571428571428571)


In [7]:
def metrics(ytrue, ypred):
    tp = fp = tn = fn = 0

    for yt, yp in zip(ytrue, ypred):
        if yt == 1 and yp == 1:
            tp += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
    
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0

    f1 = (2 *precision *recall) / (precision+recall)

    return precision,recall, f1

y_true = [1, 0, 1, 1, 0, 1]
y_pred = [1, 0, 1, 0, 0, 1]
print(metrics(y_true, y_pred)) 
# Expected TP=3, FP=0, FN=1 -> Precision=1.0, Recall=0.75, F1=0.857...


(1.0, 0.75, 0.8571428571428571)


In [12]:
import random

def train_test_split(data, test_ratio=0.2, seed=None):
    if seed is not None:
        random.seed(seed)
        
    # Create a copy so we don't mutate the original data
    shuffled_data = data[:] 
    
    # Shuffle the data
    random.shuffle(shuffled_data)
    
    # Calculate the split index
    split_idx = int(len(data) * (1 - test_ratio))
    
    # Slice the arrays
    train_set = shuffled_data[:split_idx]
    test_set = shuffled_data[split_idx:]
    
    return train_set, test_set

# Test case
dataset = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
train, test = train_test_split(dataset, test_ratio=0.2, seed=43)
print("Train:", train)
print("Test:", test)

Train: [9, 2, 6, 7, 10, 8, 4, 3]
Test: [5, 1]
